# Paper Plots: A Theoretical Analysis of Test-Driven Code Generation

This notebook loads experiment results and generates publication-ready PGF line plots.

**Prerequisites:** Run all experiments and `process_results.py` first.

In [ ]:
cd ..

In [ ]:
import os, sys

if not os.environ.get('MICROMAMBA_ENVS'):
    %env MICROMAMBA_ENVS="?/micromamba/envs"

# Ensure process_results/ is importable (it has no __init__.py)
sys.path.insert(0, 'process_results')

In [ ]:
import glob
import re
import subprocess
from math import sqrt
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patheffects as pe

from utils import read_jsonl
import plot_settings

## Display Name Mappings

In [ ]:
DATASET_MAP = {
    'leet_code_minus': 'LeetCodeDataset',
    'qiskit_human_eval': 'QiskitHumanEvalSim',
    'big_code_bench_hard': 'BigCodeBenchHard',
    'qiskit_human_eval_X': 'QiskitHumanEvalSimX',
}

MODEL_MAP = {
    'qwen3-235b-A22b-instruct': 'Qwen3 (235B-A22B-Instruct)',
    'minimax-m2.1': 'MiniMax-M2.1 (229B-A10B-Thinking)',
    'gpt-oss-120b': 'GPT-OSS (120B-A5B-Thinking)',
    'qwen3-next': 'Qwen3-Next (80B-A3B-Instruct)',
    'qwen3-next-coder': 'Qwen3-Coder-Next (80B-A3B-Instruct)',
}

MODEL_MAP_BRIEF = {
    'qwen3-235b-A22b-instruct': 'Qwen3',
    'minimax-m2.1': 'MM-M2.1',
    'gpt-oss-120b': 'GPT-OSS',
    'qwen3-next': 'Qwen3-N',
    'qwen3-next-coder': 'Qwen3-CN',
}

SELECTION_MAP = {
    'last': 'Oracle-Test Accuracy',
    'lastSGT': 'Self-Test Accuracy',
    'mbr_hard': 'MBR-Exec (Hard)',
    'mbr_soft': 'MBR-Exec (Soft)',
    'er_hard': 'Expected Reward (Hard)',
    'er_soft': 'Expected Reward (Soft)',
    'codet_hard': 'CodeT (Hard)',
    'codet_soft': 'CodeT (Soft)',
}

FEEDBACK_MAP = {
    'codes_on_tests': 'Unconditioned',
    'thompson_sampling_ind': 'Insights (Independent)',
    'thompson_sampling_tf': 'Insights (Test First)',
    'thompson_sampling_cf': 'Insights (Code First)',
    'thompson_sampling_inds': 'Summary (Independent)',
    'thompson_sampling_tfs': 'Summary (Test First)',
    'thompson_sampling_cfs': 'Summary (Code First)',
    'thompson_sampling_w_oracle': 'Insights (Oracle Tests)',
}

## Data Loading

In [ ]:
RESULTS_PATH_REGEX = re.compile(
    r"results[\\/](?P<dataset_name>[^\\/]+)[\\/](?P<model_name>[^\\/]+)"
    r"[\\/]seed-(?P<seed>\d+)[\\/]execute_(?P<gen_method>[^\\/]+)"
    r"[\\/]pass@1_(?P<selection_method>[^\\/]+)\.jsonl"
)

def load_results_data(search_pattern, n_rounds):
    """Load pass@1 results from JSONL files matching a glob pattern."""
    files = glob.glob(search_pattern)
    if not files:
        print(f"No files found matching: {search_pattern}")
        return []

    data_list = []
    print(f"Processing {len(files)} files...")
    for file_path in files:
        match = RESULTS_PATH_REGEX.search(file_path)
        if not match:
            print(f"  Skipping (regex mismatch): {file_path}")
            continue
        meta = match.groupdict()
        try:
            records = read_jsonl(file_path)
            if not records:
                continue
            data_list.append({
                'dataset_name': meta['dataset_name'],
                'model_name': meta['model_name'],
                'seed': int(meta['seed']),
                'generation_method': meta['gen_method'],
                'selection_method': meta['selection_method'],
                'pass_rates': [d['pass_rate'] * 100 for d in records][:n_rounds],
            })
        except Exception as e:
            print(f"  Error reading {file_path}: {e}")
    return data_list

## Plotting

In [ ]:
def line_plot(data, description, y_lims, *, dataset_names, model_names,
              selection_methods, feedback_methods, delta=False, show_legend=True):
    """Generate a publication-ready line plot of pass@1 across rounds."""
    assert len(dataset_names) == 1, "Multi-dataset plots not implemented"
    dataset_name = dataset_names[0]

    # Aggregate means and standard errors across seeds
    means = defaultdict(lambda: defaultdict(dict))
    sterr = defaultdict(lambda: defaultdict(dict))
    n_seeds = None

    for model in model_names:
        for sel in selection_methods:
            for fb in feedback_methods:
                rates = [
                    np.array(e['pass_rates']) for e in data
                    if e['dataset_name'] == dataset_name
                    and e['model_name'] == model
                    and e['selection_method'] == sel
                    and e['generation_method'] == fb
                ]
                if not rates:
                    continue
                if n_seeds and len(rates) != n_seeds:
                    print(f"WARNING: {model}/{sel}/{fb} has {len(rates)} seeds (expected {n_seeds})")
                else:
                    n_seeds = len(rates)
                means[model][sel][fb] = np.mean(rates, axis=0)
                sterr[model][sel][fb] = np.std(rates, axis=0, ddof=1) / sqrt(len(rates))

    # Compute global min/max for annotations
    all_means = [means[m][s][f] for m in model_names for s in selection_methods
                 for f in feedback_methods if m in means and s in means[m] and f in means[m][s]]
    first_val = round(min(a[0].item() for a in all_means), 2)
    last_val = round(max(a[-1].item() for a in all_means), 2)

    # Plot
    fig, ax = plt.subplots()
    for model in model_names:
        for sel in selection_methods:
            for fb in feedback_methods:
                if model not in means or sel not in means[model] or fb not in means[model][sel]:
                    continue
                y = means[model][sel][fb].copy()
                if delta:
                    y -= first_val
                se = sterr[model][sel][fb]
                x = np.arange(len(y)) + 1
                label = FEEDBACK_MAP.get(fb, fb)
                ax.plot(x, y, label=label, **plot_settings.format(label, label))
                ax.fill_between(x, y - se, y + se, alpha=0.2, **plot_settings.format(label, label))

    ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax.set_ylim(y_lims)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    plt.tick_params(labelbottom=True, labelleft=True)

    if delta:
        offset = y_lims[1] / 20
        ax.text(0.012, first_val + offset - (first_val if delta else 0),
                f"$\\mathbf{{{first_val}}}$\\%", transform=ax.get_yaxis_transform(),
                fontdict={'size': plt.rcParams['xtick.labelsize']}).set_path_effects(
                    [pe.withStroke(linewidth=0.75, foreground='white'), pe.Normal()])
        ax.text(0.854, last_val + offset - (first_val if delta else 0),
                f"$\\mathbf{{{last_val}}}$\\%", transform=ax.get_yaxis_transform(),
                fontdict={'size': plt.rcParams['xtick.labelsize']}).set_path_effects(
                    [pe.withStroke(linewidth=0.75, foreground='white'), pe.Normal()])

    if show_legend:
        ax.legend(loc='upper left')

    # Save PGF and compile to PDF
    plot_name = f'{dataset_name}_{model_names[0]}_ind_tfs_{description}_line_plot'
    out_dir = f'process_results/plots/{description}'
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(f'{out_dir}/{plot_name}.pgf', bbox_inches='tight')
    subprocess.run(
        ['bash', 'pgf_compiler.sh', f'plots/{description}', plot_name],
        cwd='process_results', check=True,
    )
    plt.show()

## Configure and Run

In [ ]:
from tueplots import bundles

plt.rcParams.update(bundles.icml2024(column='half', nrows=1, ncols=1))
plt.rcParams.update({'figure.figsize': (2.25, 1)})

# ── Plot configuration ──────────────────────────────────────
PLOT_CONFIG = {
    'description': 'feedback',
    'dataset_names': ['qiskit_human_eval'],
    'model_names': ['qwen3-235b-A22b-instruct'],
    'selection_methods': ['last'],
    'feedback_methods': ['thompson_sampling_tfs', 'thompson_sampling_ind'],
    'n_rounds': 10,
    'y_lims': (50, 60),
    'delta': False,
    'show_legend': True,
}

In [ ]:
search_pattern = os.path.join(
    'results', 'qiskit_human_eval', 'qwen3-*', 'seed-*',
    'execute_thompson_sampling*', 'pass@1_last*.jsonl',
)
data_list = load_results_data(search_pattern, PLOT_CONFIG['n_rounds'])

In [ ]:
line_plot(
    data_list,
    description=PLOT_CONFIG['description'],
    y_lims=PLOT_CONFIG['y_lims'],
    dataset_names=PLOT_CONFIG['dataset_names'],
    model_names=PLOT_CONFIG['model_names'],
    selection_methods=PLOT_CONFIG['selection_methods'],
    feedback_methods=PLOT_CONFIG['feedback_methods'],
    delta=PLOT_CONFIG['delta'],
    show_legend=PLOT_CONFIG['show_legend'],
)